# Automatic geometric regularization (`geometric_regularization="auto"`)

Cyclic assemblies — a homotrimer, a hexameric ring, a C9 pore — are built from subunits
that bind **head to tail**: one interface on a copy (`AA1f`) binds a *different*
interface on the next copy (`AA1b`). That is the same motif a filament uses. The ring
is simply the chain closing on itself.

Whether it closes in simulation is a geometric question. If `T` is the transform taking
subunit *k* to subunit *k+1*, the ring closes only when

$$T^{\,n} = I$$

Two things break that, and both end the same way:

1. **The per-subunit orientation was never resolved.** Every copy then carries the
   representative's frame, `T` degenerates to a pure translation, and `T**n` is never
   identity for any *n*. The design is a straight polymer.
2. **The deposited geometry is only approximately n-fold.** The closure error
   accumulates over *n* bonds until the last one falls outside NERDSS's binding
   tolerance.

Either way the chain elongates instead of closing, which shows up as **over-assembly**
(`OA`) as soon as you supply more than one copy of the deposited stoichiometry.

`geometric_regularization="auto"` detects the point group and snaps the assembly onto
it exactly. This notebook shows what it changes and — just as important — what it
refuses to touch.

In [ ]:
import logging
import numpy as np

from ionerdss.model.pdb import PDBModelBuilder
from ionerdss.model.pdb.symmetry_regularizer import SymmetryRegularizer


def frame(instance):
    "Orthonormal frame from a molecule instance's reference vectors."
    e1 = np.asarray(instance.ref1, float)
    e1 = e1 / np.linalg.norm(e1)
    raw = np.asarray(instance.ref2, float)
    e2 = raw - (raw @ e1) * e1
    e2 = e2 / np.linalg.norm(e2)
    return np.column_stack([e1, e2, np.cross(e1, e2)])


def twists(system):
    "Rotation between neighbouring subunits, in ring order. Ideal = 360/n."
    instances = list(system.molecule_instances)
    n = len(instances)
    coms = np.array([i.com for i in instances])
    centre = coms.mean(axis=0)
    axis = np.linalg.svd(coms - centre, full_matrices=False)[2][-1]
    seed = np.array([0.0, 1.0, 0.0]) if abs(np.array([1.0, 0, 0]) @ axis) > 0.9 \
        else np.array([1.0, 0.0, 0.0])
    u = seed - (seed @ axis) * axis
    u /= np.linalg.norm(u)
    v = np.cross(axis, u)
    order = np.argsort([np.arctan2((c - centre) @ v, (c - centre) @ u) for c in coms])
    frames = [frame(instances[k]) for k in order]
    out = []
    for k in range(n):
        rot = frames[(k + 1) % n] @ frames[k].T
        out.append(np.degrees(np.arccos(np.clip((np.trace(rot) - 1) / 2, -1, 1))))
    return np.array(out)


def build(pdb_id, mode):
    kwargs = {} if mode == "off" else {"geometric_regularization": mode}
    builder = PDBModelBuilder(source=pdb_id)
    return builder.build_system(
        workspace_path=f"regularization_demo/{pdb_id}_{mode}",
        generate_visualizations=False, generate_nerdss_files=False,
        logger_level=logging.ERROR, **kwargs)

## 1. A ring that is close to ideal, but not exact

`1CA4` is a homotrimer. Its three inter-subunit twists should all be 120°.

In [ ]:
before = build("1ca4", "off")
after = build("1ca4", "auto")

print("ideal for C3: 120.000")
print("off :", np.round(twists(before), 3))
print("auto:", np.round(twists(after), 3))
print("\ndetected:", after.symmetry_detection.group, "-", after.symmetry_detection.reason)

Small deviations, but they accumulate: three bonds each off by a few tenths of a degree
leave a gap the last bond has to absorb. After regularization every bond is exactly
120°, so `T**3 = I` holds and the ring can close.

## 2. A ring whose orientations were never resolved

This is the case that matters most in practice. Orientation is recovered by aligning
each copy to its group representative; when that alignment is unavailable, every copy
keeps the representative's frame.

In [ ]:
DEFAULT_REF1, DEFAULT_REF2 = np.array([1.0, 0.0, 0.0]), np.array([0.0, 0.0, 1.0])

def n_default_oriented(system):
    return sum(1 for i in system.molecule_instances
               if np.allclose(i.ref1, DEFAULT_REF1, atol=1e-6)
               and np.allclose(i.ref2, DEFAULT_REF2, atol=1e-6))

for pdb_id in ("1hg4", "1i80"):
    off, auto = build(pdb_id, "off"), build(pdb_id, "auto")
    n = len(list(off.molecule_instances))
    print(f"{pdb_id}: {n} chains, {n_default_oriented(off)}/{n} on the default frame")
    print(f"   off : {np.round(twists(off), 3)}")
    print(f"   auto: {np.round(twists(auto), 3)}")

A twist of `0.000` everywhere would mean every copy shares one orientation — the
generator is a pure translation and the "ring" is a straight polymer that can only grow.
(With a current build these come out near 120° already: the residue-intersection
alignment fix recovers the orientations upstream. `auto` then makes them exact.)

## 3. What `auto` refuses to touch

Regularization must never deform an assembly that is not actually cyclic. Two guards
do that work.

**A dihedral assembly is not a ring.** `6TLB` is a D2 tetramer whose contact graph
*is* a 4-cycle, so a naive detector would force it into C4. The fold test rejects it:
rotating the centres of mass by 90° does not map the set onto itself.

In [ ]:
d2 = build("6tlb", "auto")
print("6tlb:", d2.symmetry_detection.group, "-", d2.symmetry_detection.reason)

**A filament is meant to extend.** F-actin genuinely nucleates past the deposited
asymmetric unit, so over-assembly there is the correct answer, not an artifact. `auto`
leaves it alone.

In [ ]:
actin = build("5onv", "auto")
print("5onv:", actin.symmetry_detection.group, "-", actin.symmetry_detection.reason)

## 4. Inspecting the detection without building a full model

`SymmetryRegularizer.detect()` reports what it found and why, and never mutates the
system. Useful for triaging a set of structures before committing to a run.

In [ ]:
system = build("1ca4", "off")
detection = SymmetryRegularizer(system).detect()

print("group          :", detection.group)
print("order          :", detection.order)
print("point group    :", detection.point_group_symbol)
print("ring order     :", detection.ring)
print("regularizable  :", detection.regularizable)
print("reason         :", detection.reason)

## 5. Tuning

Two hyperparameters control the guards:

| parameter | default | effect |
|---|---|---|
| `symmetry_fold_tolerance` | `0.15` | how far from exact n-fold an assembly may be and still be accepted; this is what rejects `6TLB` |
| `com_shift_cap_ang` | `6.0` | refuse if any subunit would move further than this |

Raising the tolerance regularizes looser assemblies at the risk of forcing a
near-symmetric structure into a symmetry it does not have.

In [ ]:
loose = PDBModelBuilder(source="1ca4").build_system(
    workspace_path="regularization_demo/1ca4_loose",
    generate_visualizations=False, generate_nerdss_files=False, logger_level=logging.ERROR,
    geometric_regularization="auto",
    symmetry_fold_tolerance=0.30,
    com_shift_cap_ang=8.0,
)
print("with a looser tolerance:", loose.symmetry_detection.reason)

## When to turn it on

Use `auto` when you are supplying **more copies than the deposited stoichiometry** and
want a cyclic assembly to close rather than polymerise. At one copy it changes little:
the largest assembly that can form is the target itself, so over-assembly is
unreachable regardless.

Leave it `off` when you are studying the deposited geometry as-is, or when the
assembly is genuinely a filament and you *want* to observe extension.

Related: `ionerdss.model.pdb.structure_validation.get_free_interface_capacity()` reports
unused binding capacity before a run and predicts much of the over-assembly `auto` is
meant to prevent. See the API docs for the over-assembly preflight warning.